# 하드-네거티브 평가 (수정판) — Hard-Negative AUC, all transforms @224


In [ ]:
import os, sys, subprocess, pickle, io as _io
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

SEARCH_ROOTS=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..']
def _find(name, ftype='f', maxdepth=8):
    res=[]
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        try:
            out=subprocess.run(['find',root,'-maxdepth',str(maxdepth),'-type',ftype,'-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+= [p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_DIR=Path('./hard_negative_results'); RESULTS_DIR.mkdir(exist_ok=True)
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)

# ===== CONFIG =====
N_EVAL_CLEAN = 500          # 평가에 쓸 정상 이미지 수(가용 범위 내)
EPS = 8.0                    # 공격 ε(0..255 스케일). 매칭 노이즈 σ에 사용
UPSAMPLE_MODE = 'bicubic'   # 픽셀 도메인 업샘플(검출 파이프라인과 일치)
print('device:', device)
print('Setup done')

In [ ]:
# ── 데이터셋별 mixed_dataset.pkl 탐색 ──
# 키워드로 데이터셋 식별. 필요시 직접 경로를 PKL_PATHS에 지정하세요.
def find_mixed():
    out={}
    cands=_find('mixed_dataset.pkl')
    for p in cands:
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'imagenet_val_detector_results_eps8' in pl or ('imagenet' in pl and 'eps8' in pl): out.setdefault('ImageNet_eps8',p)
    return out

PKL_PATHS = find_mixed()   # 예: {'CIFAR-10': '.../mixed_dataset.pkl', 'ImageNet_eps8': '.../mixed_dataset.pkl'}
for k,v in PKL_PATHS.items(): print(f'✓ {k}: {v}')
if not PKL_PATHS:
    print('✗ mixed_dataset.pkl을 찾지 못했습니다. PKL_PATHS를 직접 지정하세요.')

In [ ]:
# ── 백본/SCAN/전처리 ──
CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406];   IMGNET_STD=[0.229,0.224,0.225]
def make_preprocess(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1); std=torch.tensor(s).view(1,3,1,1)
    return lambda x:(x/255.0-mean.to(x.device))/std.to(x.device)

def load_backbone(ds):
    if ds=='CIFAR-10':
        ck=(_find('resnet50_cifar10_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    elif ds=='CIFAR-100':
        ck=(_find('resnet50_cifar100_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,100)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=100
    elif ds=='SVHN':
        ck=(_find('resnet50_svhn_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    else:  # ImageNet
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2); nc=1000
    return m.to(device).eval(), nc

# SCAN 디코더
scan_dir=os.path.dirname((_find('SCAN.py') or ['.'])[0]); sys.path.insert(0,scan_dir)
from SCAN import SCAN
DECODER=(_find('scan_decoder_resnet50_imagenet.pt') or _find('scan_decoder*.pt') or [None])[0]
print('SCAN.py dir:', scan_dir); print('decoder:', DECODER)

In [ ]:
# ── 일관된 특징 함수 (정상/적대/하드-네거티브 공통, 모두 @224) ──
def gb(x, sigma):
    k=int(2*np.ceil(3*sigma)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x, kernel_size=k, sigma=sigma)

def to224(img):
    """img: [C,H,W] or [1,C,H,W] in 0..255 → [1,3,224,224] @224 (bicubic)."""
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224 or img.shape[-2]!=224:
        img=F.interpolate(img, size=(224,224), mode=UPSAMPLE_MODE, align_corners=False)
    return img.clamp(0,255)

def jpeg(img224, q):
    arr=img224.squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    buf=_io.BytesIO(); _Image.fromarray(arr).save(buf,format='JPEG',quality=int(q)); buf.seek(0)
    out=torch.from_numpy(np.array(_Image.open(buf).convert('RGB'))).float().permute(2,0,1).unsqueeze(0)
    return out.to(img224.device)

def median3(img224):
    x=F.pad(img224,(1,1,1,1),mode='reflect')
    patches=x.unfold(2,3,1).unfold(3,3,1)              # [1,C,H,W,3,3]
    return patches.contiguous().view(*patches.shape[:4],9).median(dim=-1).values

def feats(img224, backbone, preprocess, scanner, q_jpeg=75):
    """반환: dict(hfe, gl, predl1, sphf). 모두 동일 224 텐서에서."""
    with torch.no_grad():
        p0=F.softmax(backbone(preprocess(img224)),dim=1)
        # HF-Energy σ=0.5
        hfe=((img224-gb(img224,0.5)).abs().mean()/255.0).item()
        # GaussianL1 σ=1.0
        p_b=F.softmax(backbone(preprocess(gb(img224,1.0))),dim=1)
        gl=(p0-p_b).abs().sum().item()
        # PredL1 = JPEG(q) → real 3x3 median, 예측 L1
        sq=median3(jpeg(img224,q_jpeg)).clamp(0,255)
        p_sq=F.softmax(backbone(preprocess(sq)),dim=1)
        predl1=(p0-p_sq).abs().sum().item()
    # Spatial-HF-SCAN σ=0.5 (SCAN은 내부적으로 grad 사용 → no_grad 밖)
    if scanner is not None:
        c0,_=scanner(img224.squeeze(0).clamp(0,255))
        c1,_=scanner(gb(img224,0.5).squeeze(0).clamp(0,255))
        sphf=(c0.float()-c1.float()).abs().mean().item()
    else:
        sphf=float('nan')
    return {'hfe':hfe,'gl':gl,'predl1':predl1,'sphf':sphf}
print('feature fns ready')

In [ ]:
# ── 변형 정의 (모두 @224 텐서에 적용) ──
def tf_clean(x):     return x
def tf_gauss8(x):    return (x+torch.randn_like(x)*EPS).clamp(0,255)
def tf_gauss16(x):   return (x+torch.randn_like(x)*(2*EPS)).clamp(0,255)
def tf_jpeg75(x):    return jpeg(x,75)
def tf_jpeg50(x):    return jpeg(x,50)
def tf_blur1(x):     return gb(x,1.0)
HARD_TFS={'gauss_8':tf_gauss8,'gauss_16':tf_gauss16,'jpeg_75':tf_jpeg75,
          'jpeg_50':tf_jpeg50,'blur_1':tf_blur1}
print('transforms:', list(HARD_TFS.keys()))

In [ ]:
# ── 특징 추출: 정상/적대/하드-네거티브를 한 번에, 모두 @224 ──
HARD_CACHE=RESULTS_DIR/'hard_negative_features_v2.pkl'

def build_features(ds_name, pkl_path):
    backbone,nc=load_backbone(ds_name); preprocess=make_preprocess(ds_name)
    scanner=None
    if DECODER:
        scanner=SCAN(target_model=backbone,target_layer='layer4',image_size=(224,224),
                     use_gradient_mask=True,device=device,num_classes=nc)
        scanner.set_preprocess(preprocess); scanner.load_decoder(DECODER)
    with open(pkl_path,'rb') as f: mixed=pickle.load(f)
    # 원본 해상도 점검(과거 버그 진단용)
    ex=mixed[0][0]; ex=ex.squeeze(0) if ex.dim()==4 else ex
    print(f'  [{ds_name}] 저장 해상도(첫 샘플): {tuple(ex.shape)}  → 모두 224로 업샘플 후 처리')

    clean=[(im,atk) for (im,lb,atk) in mixed if atk=='clean']
    advs =[(im,atk) for (im,lb,atk) in mixed if atk!='clean']
    np.random.shuffle(clean)
    clean=clean[:N_EVAL_CLEAN]
    print(f'  clean={len(clean)}  adv={len(advs)}  attacks={sorted(set(a for _,a in advs))}')

    def extract(items, desc):
        rows=[]
        for im,atk in tqdm(items, desc=desc, leave=False):
            x=to224(im).to(device)
            fe=feats(x,backbone,preprocess,scanner); fe['attack']=atk
            rows.append(fe)
        return rows

    clean_f=extract(clean,'clean')
    adv_f  =extract(advs ,'adv')
    # 하드-네거티브: 정상 이미지를 224에서 변형 후 특징
    hard_f={name:[] for name in HARD_TFS}
    for im,_ in tqdm(clean, desc='hard-neg'):
        x=to224(im).to(device)
        for name,fn in HARD_TFS.items():
            xt=fn(x).clamp(0,255)
            hard_f[name].append(feats(xt,backbone,preprocess,scanner))
    return {'clean':clean_f,'adv':adv_f,'hard':hard_f}

if HARD_CACHE.exists():
    with open(HARD_CACHE,'rb') as f: DATA=pickle.load(f)
    print('loaded cache', HARD_CACHE)
else:
    DATA={}
    for ds,p in PKL_PATHS.items():
        print(f'\n=== {ds} ===')
        DATA[ds]=build_features(ds,p)
    with open(HARD_CACHE,'wb') as f: pickle.dump(DATA,f)
    print('saved', HARD_CACHE)

In [ ]:
# ── 평가 유틸: 캘리브레이션 분리 + bootstrap CI ──
FEATS=['hfe','gl','predl1','sphf']
FNAME={'hfe':'HF-Energy','gl':'GaussianL1','predl1':'PredL1','sphf':'Spatial-HF-SCAN'}

def arr(rows,key): return np.array([r[key] for r in rows],dtype=float)

def split_calib(clean_rows, frac=0.5, seed=SEED):
    idx=np.arange(len(clean_rows)); rng=np.random.RandomState(seed); rng.shuffle(idx)
    n=int(len(idx)*frac); return idx[:n], idx[n:]   # calib, test

def auc_ci(neg, pos, n_boot=1000, seed=SEED):
    """neg, pos: 이미 anomaly score로 변환된 1D 배열."""
    y=np.r_[np.zeros(len(neg)),np.ones(len(pos))]; s=np.r_[neg,pos]
    if len(set(y))<2: return float('nan'),float('nan'),float('nan')
    base=roc_auc_score(y,s)
    rng=np.random.RandomState(seed); boots=[]
    N=len(y)
    for _ in range(n_boot):
        ii=rng.randint(0,N,N)
        if len(set(y[ii]))<2: continue
        boots.append(roc_auc_score(y[ii],s[ii]))
    lo,hi=np.percentile(boots,[2.5,97.5]) if boots else (float('nan'),float('nan'))
    return base,lo,hi

def feature_scores(clean_rows, eval_neg_rows, pos_rows, key):
    """calib clean으로 μ/σ 추정 → eval_neg, pos를 anomaly=|z|로."""
    ci,ti=split_calib(clean_rows)
    cf=arr(clean_rows,key); mu=cf[ci].mean(); sd=cf[ci].std()+1e-8
    def anom(rows): return np.abs((arr(rows,key)-mu)/sd)
    neg=np.r_[anom([clean_rows[i] for i in ti]), anom(eval_neg_rows)]
    pos=anom(pos_rows)
    return neg,pos

def ensemble_scores(clean_rows, eval_neg_rows, pos_rows, keys):
    ci,ti=split_calib(clean_rows)
    mus={k:arr(clean_rows,k)[ci].mean() for k in keys}
    sds={k:arr(clean_rows,k)[ci].std()+1e-8 for k in keys}
    def ens(rows):
        z=np.mean([ (arr(rows,k)-mus[k])/sds[k] for k in keys ],axis=0)
        return z
    # 앙상블 중심도 calib clean에서
    ce=ens([clean_rows[i] for i in ci]); mu_e=ce.mean()
    def anom(rows): return np.abs(ens(rows)-mu_e)
    neg=np.r_[anom([clean_rows[i] for i in ti]), anom(eval_neg_rows)]
    pos=anom(pos_rows)
    return neg,pos
print('eval utils ready')

In [ ]:
# ── Protocol A: 확장 네거티브 ΔAUC (모든 특징 + 앙상블) ──
TF_GROUPS={'Pristine only':[], '+Matched noise σ=8':['gauss_8'], '+JPEG q=75':['jpeg_75'],
           '+Blur σ=1':['blur_1'], '+All mixed':['gauss_8','gauss_16','jpeg_75','jpeg_50','blur_1']}
ENS_KEYS=['hfe','gl','predl1']

print('='*92); print('  Protocol A — 확장 네거티브 AUROC (calib 50% clean / test 50%), 95% CI'); print('='*92)
for ds,d in DATA.items():
    clean,adv,hard=d['clean'],d['adv'],d['hard']
    print(f'\n  [{ds}]  (clean={len(clean)}, adv={len(adv)})')
    cols=[FNAME[k] for k in FEATS]+['Ensemble']
    print('  '+f"{'Neg 구성':<22}"+''.join(f'{c:>20}' for c in cols))
    base={}
    for grp,tfs in TF_GROUPS.items():
        neg_hard={k:[r for t in tfs for r in hard[t]] for k in FEATS}
        row=f'  {grp:<22}'
        for k in FEATS:
            neg,pos=feature_scores(clean, neg_hard[k], adv, k)
            a,lo,hi=auc_ci(neg,pos)
            row+=f'{a:>9.4f}[{lo:.3f},{hi:.3f}]'.rjust(20)
            base.setdefault(k,{})[grp]=a
        negc=[r for t in tfs for r in hard[t]]
        neg,pos=ensemble_scores(clean, negc, adv, ENS_KEYS)
        a,lo,hi=auc_ci(neg,pos); base.setdefault('ens',{})[grp]=a
        row+=f'{a:>9.4f}[{lo:.3f},{hi:.3f}]'.rjust(20)
        print(row)
    print('  '+'-'*100)
    row=f"  {'ΔAUC (mixed-pristine)':<22}"
    for k in FEATS+['ens']:
        d_=base[k]['+All mixed']-base[k]['Pristine only']
        row+=f'{d_:>+20.4f}'
    print(row)

In [ ]:
# ── Protocol B: 쌍대 진단 (적대적 vs 단일 변형) ──
print('='*92); print('  Protocol B — 적대적 vs 단일 변형 AUROC (낮을수록 그 변형과 구별 못함)'); print('='*92)
for ds,d in DATA.items():
    clean,adv,hard=d['clean'],d['adv'],d['hard']
    atks=sorted(set(r['attack'] for r in adv))
    print(f'\n  [{ds}]  공격: {atks}')
    for tf in ['gauss_8','jpeg_75','blur_1']:
        if tf not in hard: continue
        print(f'\n  [적대적 vs {tf}]')
        print('  '+f"{'특징':<16}{'All-adv':>10}"+''.join(f'{a:>10}' for a in atks))
        for k in FEATS:
            negh=hard[tf]
            neg,pos=feature_scores(clean,negh,adv,k); a_all,_,_=auc_ci(neg,pos,n_boot=200)
            row=f'  {FNAME[k]:<16}{a_all:>10.4f}'
            for at in atks:
                pos_a=[r for r in adv if r['attack']==at]
                neg,pos=feature_scores(clean,negh,pos_a,k); a,_,_=auc_ci(neg,pos,n_boot=200)
                row+=f'{a:>10.4f}'
            print(row)
    print()
print('핵심: C&W vs Matched-noise 에서 GaussianL1/PredL1 AUC가 높으면 → 예측압축이 적대성을 포착')

In [ ]:
# ── 요약 + 논문 표 H1 출력 + 저장 ──
print('='*80); print('  요약 — 표 H1 (논문 5.8 Protocol A) 채움값'); print('='*80)
summary={}
for ds,d in DATA.items():
    clean,adv,hard=d['clean'],d['adv'],d['hard']
    print(f'\n  [{ds}]'); print(f"  {'특징':<16}{'Pristine':>10}{'+All mixed':>12}{'ΔAUC':>10}  결론")
    summary[ds]={}
    for k in FEATS+['ens']:
        if k=='ens':
            neg0,pos0=ensemble_scores(clean,[],adv,ENS_KEYS)
            negm=[r for t in ['gauss_8','gauss_16','jpeg_75','jpeg_50','blur_1'] for r in hard[t]]
            negM,posM=ensemble_scores(clean,negm,adv,ENS_KEYS); name='Ensemble'
        else:
            neg0,pos0=feature_scores(clean,[],adv,k)
            negm=[r for t in ['gauss_8','gauss_16','jpeg_75','jpeg_50','blur_1'] for r in hard[t]]
            negM,posM=feature_scores(clean,negm,adv,k); name=FNAME[k]
        p,_,_=auc_ci(neg0,pos0,n_boot=200); m,_,_=auc_ci(negM,posM,n_boot=200); dlt=m-p
        verdict=('붕괴' if dlt<-0.05 else ('유지' if abs(dlt)<0.02 else '감소'))
        print(f'  {name:<16}{p:>10.4f}{m:>12.4f}{dlt:>+10.4f}  {verdict}')
        summary[ds][name]={'pristine':p,'mixed':m,'delta':dlt}
with open(RESULTS_DIR/'hard_negative_summary_v2.pkl','wb') as f: pickle.dump(summary,f)
print('\n저장:', RESULTS_DIR/'hard_negative_summary_v2.pkl')
print("""
해석 가이드:
  HF↓ & (GaussL/PredL/Ensemble)↑  →  '앙상블이 진짜 적대성 포착' (논문 시나리오 1, 기여 격상)
  전부 ↓                          →  '전처리 없는 파이프라인 전용 섭동 탐지기' (시나리오 2, 범위 한정)
이 표를 paper v6 의 5.8 Protocol A / Protocol B 빈칸에 기입하세요.""")